# Bilge Pump System — Verification Results

Evaluates all analysis cases defined in `Analysis.ipynb` and renders colored pass/fail tables.

| Item | Value |
|---|---|
| Kernel | Python 3 |
| Model source | `Analysis.ipynb` (SysML v2 kernel) |
| No external deps | Uses only `IPython.display` (stdlib) |

## Scenarios

| Cell | Scenario | Source |
|---|---|---|
| 3 | Positive test — nominal values | Cell 5 of Analysis.ipynb |
| 4 | Negative test — pump A offline | Cell 6 of Analysis.ipynb |
| 5 | Pipe-loss sensitivity sweep | Phase B of Analysis.ipynb |
| 6 | FMEA fault matrix — 3 scenarios × 4 requirements | Phase C of Analysis.ipynb |
| 7 | Master summary — all scenarios in one view | Combined |

In [1]:
# ============================================================
# Cell 2 — Shared setup: parameters + HTML rendering helpers
# ============================================================
from IPython.display import display, HTML

# ── Color constants ──────────────────────────────────────────
GREEN  = '#d4edda'   # SATISFIED / PASS
RED    = '#f8d7da'   # VIOLATED / FAIL
YELLOW = '#fff3cd'   # warning / partial
HEADER = '#343a40'   # dark header background
WHITE  = '#ffffff'

# ── Physics engine ───────────────────────────────────────────
def q_net(flow_a, flow_b, eta=0.82, lam=0.05):
    """Q_net = (Q_A + Q_B) × η × (1 − λ)"""
    return (flow_a + flow_b) * eta * (1.0 - lam)

# ── Requirement evaluator ────────────────────────────────────
def check_requirements(water_level, is_redundant, alarm_delay, q, design_inflow=0.030):
    return [
        ('BPS-REQ-001', 'waterLevel ≤ 0.30 m',     f'{water_level:.2f} m',      water_level   <= 0.30),
        ('BPS-REQ-002', 'pumpB.isRedundant',        str(is_redundant),           is_redundant  == True),
        ('BPS-REQ-003', 'alarmDelay ≤ 2.00 s',     f'{alarm_delay:.2f} s',      alarm_delay   <= 2.0),
        ('BPS-REQ-004', f'Q_net ≥ {design_inflow}', f'{q:.4f} m³/s',            q             >= design_inflow),
    ]

# ── CSS shared style ─────────────────────────────────────────
BASE_CSS = """
<style>
  .bp-table { border-collapse: collapse; font-family: monospace; font-size: 13px;
               margin: 8px 0; min-width: 560px; }
  .bp-table th { background: #343a40; color: #fff; padding: 7px 12px;
                  text-align: left; font-weight: bold; }
  .bp-table td { padding: 6px 12px; border: 1px solid #dee2e6; }
  .bp-pass   { background: #d4edda; color: #155724; font-weight: bold; }
  .bp-fail   { background: #f8d7da; color: #721c24; font-weight: bold; }
  .bp-warn   { background: #fff3cd; color: #856404; font-weight: bold; }
  .bp-title  { font-family: monospace; font-weight: bold; font-size: 14px;
               margin-top: 12px; padding: 5px 0; }
  .bp-eq     { font-family: monospace; color: #495057; font-size: 12px;
               margin: 2px 0 6px 0; }
</style>
"""

def verdict_cell(satisfied):
    if satisfied:
        return '<td class="bp-pass">SATISFIED ✓</td>'
    return '<td class="bp-fail">VIOLATED ✗</td>'

def render_scenario(title, equation, checks):
    """Render one scenario as a colored 4-row requirement table."""
    all_pass = all(sat for _, _, _, sat in checks)
    overall_cls = 'bp-pass' if all_pass else 'bp-fail'
    overall_txt = 'ALL SATISFIED ✓' if all_pass else 'FAILURES DETECTED ✗'

    rows = ''.join(
        f'<tr><td>{req}</td><td>{desc}</td><td>{val}</td>{verdict_cell(sat)}</tr>'
        for req, desc, val, sat in checks
    )
    html = f"""
    {BASE_CSS}
    <div class="bp-title">{title}</div>
    <div class="bp-eq">{equation}</div>
    <table class="bp-table">
      <thead><tr>
        <th>Requirement</th><th>Constraint</th><th>Value</th><th>Result</th>
      </tr></thead>
      <tbody>{rows}</tbody>
      <tfoot><tr>
        <td colspan="3" style="text-align:right;font-weight:bold;">Overall</td>
        <td class="{overall_cls}">{overall_txt}</td>
      </tr></tfoot>
    </table>"""
    display(HTML(html))

print('Setup complete — helpers ready.')

Setup complete — helpers ready.


In [2]:
# ============================================================
# Cell 3 — POSITIVE TEST: Nominal values
#
# Mirror of Analysis.ipynb Cell 5 (positiveTest)
# Q_net = (0.025 + 0.025) × 0.82 × (1 − 0.05) = 0.0389 m³/s
# ============================================================

fa, fb, eta, lam = 0.025, 0.025, 0.82, 0.05
q = q_net(fa, fb, eta, lam)

checks = check_requirements(
    water_level   = 0.15,
    is_redundant  = True,
    alarm_delay   = 0.5,
    q             = q,
)

render_scenario(
    title    = 'POSITIVE TEST — Nominal values',
    equation = f'Q_net = ({fa} + {fb}) × {eta} × (1 − {lam}) = <b>{q:.4f} m³/s</b>',
    checks   = checks,
)

In [7]:
# ============================================================
# Cell 4 — NEGATIVE TEST: Pump A offline
#
# Mirror of Analysis.ipynb Cell 6 (negativeTest)
# Q_net = (0.0 + 0.025) × 0.82 × 0.95 = 0.0195 m³/s < 0.030
# ============================================================

fa, fb, eta, lam = 0.0, 0.025, 0.82, 0.05
q = q_net(fa, fb, eta, lam)

checks = check_requirements(
    water_level   = 0.15,
    is_redundant  = True,
    alarm_delay   = 0.5,
    q             = q,
)

render_scenario(
    title    = 'NEGATIVE TEST — Pump A offline (pumpAFlowRate = 0.0)',
    equation = f'Q_net = (0.0 + {fb}) × {eta} × (1 − {lam}) = <b>{q:.4f} m³/s</b>',
    checks   = checks,
)

In [8]:
# ============================================================
# Cell 5 — PIPE-LOSS SENSITIVITY SWEEP
#
# Mirror of Analysis.ipynb Phase B (BilgePump_Parametric)
# λ swept from 0% to 20%. Breakeven ≈ 26.8%.
# ============================================================

fa = fb = 0.025
eta = 0.82
design = 0.030
sweep_points = [0.00, 0.05, 0.10, 0.15, 0.20]

breakeven = 1.0 - design / ((fa + fb) * eta)

rows = ''
for lam in sweep_points:
    q = q_net(fa, fb, eta, lam)
    sat = q >= design
    cls = 'bp-pass' if sat else 'bp-fail'
    verdict = 'PASS ✓' if sat else 'FAIL ✗'
    rows += (f'<tr>'
             f'<td>{lam*100:.0f}%</td>'
             f'<td>{q:.4f} m³/s</td>'
             f'<td class="{cls}">{verdict}</td>'
             f'</tr>')

html = f"""
{BASE_CSS}
<div class="bp-title">PIPE-LOSS SENSITIVITY SWEEP — Q_A = Q_B = 0.025 m³/s, η = 0.82</div>
<div class="bp-eq">Q_net = (Q_A + Q_B) × η × (1 − λ) &nbsp;|&nbsp; Design inflow threshold: 0.030 m³/s</div>
<table class="bp-table">
  <thead><tr>
    <th>λ (pipe loss)</th><th>Q_net (m³/s)</th><th>BPS-REQ-004</th>
  </tr></thead>
  <tbody>{rows}</tbody>
  <tfoot><tr>
    <td colspan="3" style="font-family:monospace;padding:6px 12px;
        background:#f8f9fa;">
      Breakeven λ = <b>{breakeven:.3f}</b> &nbsp;({breakeven*100:.1f}% pipe loss → BPS-REQ-004 violated)
    </td>
  </tr></tfoot>
</table>"""
display(HTML(html))

In [9]:
# ============================================================
# Cell 6 — FMEA FAULT MATRIX
#
# Mirror of Analysis.ipynb Phase C (BilgePump_FMEA)
# Grid: 3 fault scenarios × 4 requirements
# ============================================================

design = 0.030
eta = 0.82
lam = 0.05

faults = [
    ('Fault A\nPump A offline',       0.0,   0.025, 0.15, True, 0.5),
    ('Fault B\nBoth pumps offline',   0.0,   0.0,   0.15, True, 0.5),
    ('Fault C\nSensor stuck-at-zero', 0.025, 0.025, 0.0,  True, 0.5),
]
req_labels = ['BPS-REQ-001\nwaterLevel ≤ 0.30 m',
              'BPS-REQ-002\npumpB.isRedundant',
              'BPS-REQ-003\nalarmDelay ≤ 2.00 s',
              'BPS-REQ-004\nQ_net ≥ design']

header_cells = ''.join(f'<th style="white-space:pre">{r}</th>' for r in req_labels)

body_rows = ''
for label, fa, fb, wl, red, ad in faults:
    q = q_net(fa, fb, eta, lam)
    checks = check_requirements(wl, red, ad, q, design)
    result_cells = ''.join(verdict_cell(sat) for _, _, _, sat in checks)
    body_rows += f'<tr><td style="white-space:pre;font-weight:bold">{label}</td>{result_cells}</tr>'

html = f"""
{BASE_CSS}
<div class="bp-title">FMEA — Fault Scenario × Requirement Matrix</div>
<div class="bp-eq">η = {eta} &nbsp;|&nbsp; λ = {lam} &nbsp;|&nbsp; Design inflow = {design} m³/s</div>
<table class="bp-table">
  <thead><tr>
    <th>Scenario</th>{header_cells}
  </tr></thead>
  <tbody>{body_rows}</tbody>
</table>
<div class="bp-eq" style="margin-top:6px">
  SOLAS II-1 Reg. 35: Fault A confirms pump B alone (0.0195 m³/s) is below design inflow.
  Pump B must be upsized or a 3rd pump added.
</div>"""
display(HTML(html))

Scenario,BPS-REQ-001 waterLevel ≤ 0.30 m,BPS-REQ-002 pumpB.isRedundant,BPS-REQ-003 alarmDelay ≤ 2.00 s,BPS-REQ-004 Q_net ≥ design
Fault A Pump A offline,SATISFIED ✓,SATISFIED ✓,SATISFIED ✓,VIOLATED ✗
Fault B Both pumps offline,SATISFIED ✓,SATISFIED ✓,SATISFIED ✓,VIOLATED ✗
Fault C Sensor stuck-at-zero,SATISFIED ✓,SATISFIED ✓,SATISFIED ✓,SATISFIED ✓


In [10]:
# ============================================================
# Cell 7 — MASTER SUMMARY: all 6 scenarios in one view
# ============================================================

design = 0.030
eta    = 0.82
lam    = 0.05

scenarios = [
    ('Positive test (nominal)',     0.025, 0.025, 0.15, True, 0.5),
    ('Negative test (pump A off)',  0.0,   0.025, 0.15, True, 0.5),
    ('FMEA Fault A (pump A off)',   0.0,   0.025, 0.15, True, 0.5),
    ('FMEA Fault B (both off)',     0.0,   0.0,   0.15, True, 0.5),
    ('FMEA Fault C (sensor stuck)', 0.025, 0.025, 0.0,  True, 0.5),
    ('Sweep λ=20% (degraded pipe)', 0.025, 0.025, 0.15, True, 0.5),
]

# λ override for sweep scenario
sweep_lam = {scenarios[5][0]: 0.20}

body_rows = ''
for label, fa, fb, wl, red, ad in scenarios:
    l = sweep_lam.get(label, lam)
    q = q_net(fa, fb, eta, l)
    checks = check_requirements(wl, red, ad, q, design)
    all_pass = all(sat for _, _, _, sat in checks)
    overall_cls = 'bp-pass' if all_pass else 'bp-fail'
    overall_txt = 'ALL SATISFIED ✓' if all_pass else 'VIOLATED ✗'
    req_cells = ''.join(verdict_cell(sat) for _, _, _, sat in checks)
    q_cell = f'<td style="font-family:monospace">{q:.4f}</td>'
    body_rows += (f'<tr>'
                  f'<td style="font-weight:bold">{label}</td>'
                  f'{q_cell}'
                  f'{req_cells}'
                  f'<td class="{overall_cls}">{overall_txt}</td>'
                  f'</tr>')

html = f"""
{BASE_CSS}
<div class="bp-title">MASTER VERIFICATION SUMMARY — Bilge Pump System</div>
<div class="bp-eq">η = {eta} &nbsp;|&nbsp; design inflow = {design} m³/s &nbsp;|&nbsp; nominal λ = {lam}</div>
<table class="bp-table">
  <thead><tr>
    <th>Scenario</th>
    <th>Q_net (m³/s)</th>
    <th>REQ-001</th><th>REQ-002</th><th>REQ-003</th><th>REQ-004</th>
    <th>Overall</th>
  </tr></thead>
  <tbody>{body_rows}</tbody>
</table>"""
display(HTML(html))

Scenario,Q_net (m³/s),REQ-001,REQ-002,REQ-003,REQ-004,Overall
Positive test (nominal),0.0389,SATISFIED ✓,SATISFIED ✓,SATISFIED ✓,SATISFIED ✓,ALL SATISFIED ✓
Negative test (pump A off),0.0195,SATISFIED ✓,SATISFIED ✓,SATISFIED ✓,VIOLATED ✗,VIOLATED ✗
FMEA Fault A (pump A off),0.0195,SATISFIED ✓,SATISFIED ✓,SATISFIED ✓,VIOLATED ✗,VIOLATED ✗
FMEA Fault B (both off),0.0000,SATISFIED ✓,SATISFIED ✓,SATISFIED ✓,VIOLATED ✗,VIOLATED ✗
FMEA Fault C (sensor stuck),0.0389,SATISFIED ✓,SATISFIED ✓,SATISFIED ✓,SATISFIED ✓,ALL SATISFIED ✓
Sweep λ=20% (degraded pipe),0.0328,SATISFIED ✓,SATISFIED ✓,SATISFIED ✓,SATISFIED ✓,ALL SATISFIED ✓
